In [1]:
import numpy as np

Пересчёт геометрических параметров

In [2]:
def geometric_parameters(scheme, l_1, l_2, l_3):
    match scheme:
        case 1:
            return l_1, l_2, l_3
        case 2:
            return l_1, 0, l_2
        case 3:
            return l_2, l_3, l_1
        case 4:
            return l_2, 0, l_1
        case 5:
            return l_1, l_3, l_2
        case 6:
            return -l_1, l_3-l_1, l_2-l_1

Расчёт реакций

In [3]:
def reactions(a, b, c, F_1, F_2):
    R_1 = (a/c-1)*F_1+(b/c-1)*F_2
    R_2 = -a/c*F_1-b/c*F_2
    return R_1, R_2

Расчёт сил (Сапфир)

In [4]:
def forces_sapphire(m_1, m_2, omega, r_1, r_2):
    F_1 = m_1*omega*r_1
    F_2 = m_2*omega*r_2
    return F_1, F_2

Расчёт сил

In [5]:
def forces(m_1, m_2, omega, r_1, r_2):
    F_1 = m_1*omega**2*r_1
    F_2 = m_2*omega**2*r_2
    return F_1, F_2

Функция вывода матрицы

In [6]:
def print_matrix(matrix):
    m_11_amplitude = np.abs(matrix[0][0])
    m_11_phase = np.angle(matrix[0][0], deg=True)
    m_12_amplitude = np.abs(matrix[0][1])
    m_12_phase = np.angle(matrix[0][1], deg=True)
    m_21_amplitude = np.abs(matrix[1][0])
    m_21_phase = np.angle(matrix[1][0], deg=True)
    m_22_amplitude = np.abs(matrix[1][1])
    m_22_phase = np.angle(matrix[1][1], deg=True)

    if m_11_phase < 0:
        m_11_phase += 360
    if m_12_phase < 0:
        m_12_phase += 360
    if m_21_phase < 0:
        m_21_phase += 360
    if m_22_phase < 0:
        m_22_phase += 360
    print(f'{m_11_amplitude:.4f}, {m_11_phase:.2f}     {m_12_amplitude:.4f}, {m_12_phase:.2f}')
    print(f'{m_21_amplitude:.4f}, {m_21_phase:.2f}     {m_22_amplitude:.4f}, {m_22_phase:.2f}')

Функция задания комплесного числа по амплитуде и фазе в градусах

In [7]:
def get_complex(amp, pha):
    return amp*np.cos(np.deg2rad(pha))+1j*amp*np.sin(np.deg2rad(pha))

## Расчёт эталона


In [8]:
l_1 = 60
l_2 = 180
l_3 = 235
k=1e-3
r_1 = 60*k
r_2 = 60*k

omega = 1000*2*np.pi/60

scheme=1

Плоскость 1

In [9]:
m_1 = get_complex(1, 0)
m_2 = get_complex(0, 0)

a, b, c = geometric_parameters(scheme=scheme, l_1=l_1, l_2=l_2, l_3=l_3)
F_1, F_2 = forces_sapphire(m_1, m_2, omega, r_1, r_2)
R_11, R_21 = reactions(a, b, c, F_1, F_2)

Плоскость 2

In [10]:
m_1 = get_complex(0, 0)
m_2 = get_complex(1, 0)

a, b, c = geometric_parameters(scheme=scheme, l_1=l_1, l_2=l_2, l_3=l_3)
F_1, F_2 = forces_sapphire(m_1, m_2, omega, r_1, r_2)
R_12, R_22 = reactions(a, b, c, F_1, F_2)

Матрица R

In [11]:
matrix_R = [[R_11, R_12], [R_21, R_22]]
print_matrix(matrix_R)

4.6790, 180.00     1.4705, 180.00
1.6042, 180.00     4.8127, 180.00


Коэффициенты S с прибора

In [12]:
S_11 = get_complex(16.91, 351)
S_21 = get_complex(5.19, 350)
S_12 = get_complex(5.58, 351)
S_22 = get_complex(15.08, 351)
matrix_S = [[S_11, S_12], [S_21, S_22]]
print_matrix(matrix_S)

16.9100, 351.00     5.5800, 351.00
5.1900, 350.00     15.0800, 351.00


Матрица P

In [13]:
matrix_P = np.dot(matrix_S, np.linalg.inv(matrix_R))
print_matrix(matrix_P)


3.5929, 171.00     0.0616, 171.00
0.0444, 141.87     3.1216, 171.12


Проверка матрицы S

In [14]:
print_matrix(np.dot(matrix_P, matrix_R))

16.9100, 351.00     5.5800, 351.00
5.1900, 350.00     15.0800, 351.00


## Произвольный ротор

Вариант 1-1

In [15]:
l_1 = 65
l_2 = 255
l_3 = 635

k=1e-3
r_1 = 100*k
r_2 = 100*k
omega = 500*2*np.pi/60

m_1 = get_complex(1, 0)
m_2 = get_complex(0, 0)

a, b, c = geometric_parameters(scheme=1, l_1=l_1, l_2=l_2, l_3=l_3)
F_1, F_2 = forces_sapphire(m_1, m_2, omega, r_1, r_2)
R_11, R_21 = reactions(a, b, c, F_1, F_2)

m_1 = get_complex(0, 0)
m_2 = get_complex(1, 0)
a, b, c = geometric_parameters(scheme=1, l_1=l_1, l_2=l_2, l_3=l_3)
F_1, F_2 = forces_sapphire(m_1, m_2, omega, r_1, r_2)
R_12, R_22 = reactions(a, b, c, F_1, F_2)

matrix_R = [[R_11, R_12], [R_21, R_22]]
matrix_S = np.dot(matrix_P, matrix_R)
print_matrix(matrix_S)

16.9198, 351.00     11.3874, 351.00
1.8580, 347.97     6.6853, 350.54


Вариант 5-3

In [16]:
l_1 = 225
l_2 = 420
l_3 = 520

k=1e-3
r_1 = 100*k
r_2 = 100*k
omega = 500*2*np.pi/60

scheme=5

m_1 = get_complex(1, 0)
m_2 = get_complex(0, 0)

a, b, c = geometric_parameters(scheme=scheme, l_1=l_1, l_2=l_2, l_3=l_3)
F_1, F_2 = forces_sapphire(m_1, m_2, omega, r_1, r_2)
R_11, R_21 = reactions(a, b, c, F_1, F_2)

m_1 = get_complex(0, 0)
m_2 = get_complex(1, 0)
a, b, c = geometric_parameters(scheme=1, l_1=l_1, l_2=l_2, l_3=l_3)
F_1, F_2 = forces_sapphire(m_1, m_2, omega, r_1, r_2)
R_12, R_22 = reactions(a, b, c, F_1, F_2)

matrix_R = [[R_11, R_12], [R_21, R_22]]
matrix_S = np.dot(matrix_P, matrix_R)
print_matrix(matrix_S)

8.9072, 351.00     3.8783, 351.00
8.8503, 350.78     13.2403, 351.03
